In [9]:
import sys
import os

sys.path.append(os.path.abspath("../src"))

import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error

# Load data
train = pd.read_csv("../data/processed/train.csv")
val = pd.read_csv("../data/processed/validation.csv")
test = pd.read_csv("../data/processed/test.csv")

print("Train:", train.shape)

Train: (80000, 4)


In [10]:
from hybrid import HybridRecommender

In [11]:
# ---- Train Bias Model ----
from baseline import BiasModel

bias_model = BiasModel()
bias_model.fit(train)


# ---- Train UserKNN ----
from knn import UserKNN

user_knn_model = UserKNN(k=20)
user_knn_model.fit(train)


# ---- Train ItemKNN ----
from knn import ItemKNN

item_knn_model = ItemKNN(k=40)
item_knn_model.fit(train)

In [12]:
hybrid_model = HybridRecommender(
    bias_model=bias_model,
    user_knn_model=user_knn_model,
    item_knn_model=item_knn_model,
    alpha=0.3,
    beta=0.4
)

val_preds_hybrid = hybrid_model.predict(val)

rmse_hybrid = np.sqrt(mean_squared_error(val["rating"], val_preds_hybrid))

print("Validation RMSE (Hybrid):", round(rmse_hybrid, 4))

Validation RMSE (Hybrid): 0.9325


In [13]:
best_rmse = float("inf")
best_params = None

for alpha in np.arange(0.1, 0.6, 0.1):
    for beta in np.arange(0.1, 0.6, 0.1):

        if alpha + beta >= 1:
            continue

        model = HybridRecommender(
            bias_model=bias_model,
            user_knn_model=user_knn_model,
            item_knn_model=item_knn_model,
            alpha=alpha,
            beta=beta
        )

        preds = model.predict(val)
        rmse = np.sqrt(mean_squared_error(val["rating"], preds))

        print(f"alpha={alpha:.1f}, beta={beta:.1f}, RMSE={rmse:.4f}")

        if rmse < best_rmse:
            best_rmse = rmse
            best_params = (alpha, beta)

print("\nBest parameters:", best_params)
print("Best RMSE:", round(best_rmse, 4))

alpha=0.1, beta=0.1, RMSE=0.9563
alpha=0.1, beta=0.2, RMSE=0.9458
alpha=0.1, beta=0.3, RMSE=0.9377
alpha=0.1, beta=0.4, RMSE=0.9321
alpha=0.1, beta=0.5, RMSE=0.9290
alpha=0.2, beta=0.1, RMSE=0.9485
alpha=0.2, beta=0.2, RMSE=0.9402
alpha=0.2, beta=0.3, RMSE=0.9343
alpha=0.2, beta=0.4, RMSE=0.9309
alpha=0.2, beta=0.5, RMSE=0.9301
alpha=0.3, beta=0.1, RMSE=0.9433
alpha=0.3, beta=0.2, RMSE=0.9372
alpha=0.3, beta=0.3, RMSE=0.9336
alpha=0.3, beta=0.4, RMSE=0.9325
alpha=0.3, beta=0.5, RMSE=0.9340
alpha=0.4, beta=0.1, RMSE=0.9409
alpha=0.4, beta=0.2, RMSE=0.9370
alpha=0.4, beta=0.3, RMSE=0.9357
alpha=0.4, beta=0.4, RMSE=0.9369
alpha=0.4, beta=0.5, RMSE=0.9406
alpha=0.5, beta=0.1, RMSE=0.9412
alpha=0.5, beta=0.2, RMSE=0.9396
alpha=0.5, beta=0.3, RMSE=0.9405
alpha=0.5, beta=0.4, RMSE=0.9440

Best parameters: (np.float64(0.1), np.float64(0.5))
Best RMSE: 0.929


In [14]:
best_alpha, best_beta = best_params

final_hybrid = HybridRecommender(
    bias_model=bias_model,
    user_knn_model=user_knn_model,
    item_knn_model=item_knn_model,
    alpha=best_alpha,
    beta=best_beta
)

test_preds = final_hybrid.predict(test)
test_rmse = np.sqrt(mean_squared_error(test["rating"], test_preds))

print("Final Test RMSE (Hybrid):", round(test_rmse, 4))

Final Test RMSE (Hybrid): 0.9291
